In [1]:
%matplotlib inline

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import matplotlib.pyplot as plt
import gymnasium as gym
import jax
import jax.numpy as jnp
import optax

from jax_mc_pilco.model_learning.flow_model import FlowDynamics
from jax_mc_pilco.policy_learning.action_flows import FlowActor
from jax_mc_pilco.training.learning import collect_experience, train_actor, train_flow, train_reward

jax.config.update("jax_enable_x64", True)

In [4]:
env = gym.make("InvertedDoublePendulum-v5")  # gym.make("InvertedPendulum-v5")

key = jax.random.key(seed=4)
key, subkey = jax.random.split(key)
_, _, states, actions, next_states, rewards = collect_experience(env, 4096, subkey, actor=None, exploration=True, use_sobol=True)

In [5]:
states.shape, actions.shape, next_states.shape, rewards.shape

((4096, 9), (4096, 1), (4096, 9), (4096, 1))

In [6]:
key, subkey = jax.random.split(key)
dynamics, losses = train_flow(states, actions, next_states, subkey)

  9%|██▋                            | 87/1000 [00:07<01:18, 11.68it/s, train=4.23e+6, val=38.7 (Max patience reached)]


In [7]:
key, subkey = jax.random.split(key)
reward_fn = train_reward(states, actions, rewards, subkey)

Beginning SVGP Mini-batched Optimization for 1600 iterations...


  0%|          | 0/1600 [00:00<?, ?it/s]

In [8]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

act_low = jnp.array(env.action_space.low, dtype=float)
act_high = jnp.array(env.action_space.high, dtype=float)
key, actor_key = jax.random.split(key)

actor = FlowActor(
    key=actor_key,
    state_dim=state_dim,
    action_dim=action_dim,
    action_low=act_low,
    action_high=act_high,
    flow_layers=4,
)

In [9]:
optim = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(3e-4),
)

key, subkey = jax.random.split(key)
actor = train_actor(actor, dynamics, states, subkey, optim, reward_fn, num_train_steps=250)

  [actor] step 0, loss -200.8870
  [actor] step 20, loss -200.8686
  [actor] step 40, loss -200.9446
  [actor] step 60, loss -200.9029
  [actor] step 80, loss -200.9826
  [actor] step 100, loss -200.9504
  [actor] step 120, loss -200.9070
  [actor] step 140, loss -200.9418
  [actor] step 160, loss -200.9130
  [actor] step 180, loss -200.9980
  [actor] step 200, loss -200.9773
  [actor] step 220, loss -200.9051
  [actor] step 240, loss -200.9677


In [10]:
def diagnose_model_reality_gap(
    dynamics: FlowDynamics,
    s_curr_ep: jax.Array,
    action_ep: jax.Array,
    s_next_ep: jax.Array,
    key: jax.Array,
    num_samples: int = 20,
) -> None:
    """Prints, for each real transition in the episode, the model's
    log_prob of the real outcome and the discrepancy between a model
    *sample* and what actually happened. Large negative log_prob or large
    sample discrepancy on the very states the actor's rollout visited is
    direct evidence the actor is exploiting model error rather than
    learning real stabilizing behavior."""
    for i in range(s_curr_ep.shape[0]):
        s_c, a, s_n = s_curr_ep[i], action_ep[i], s_next_ep[i]
        lp = dynamics.log_prob(s_n, jnp.concatenate([s_c, a], axis=-1))

        keys = jax.random.split(key, num_samples + 1)
        key = keys[0]
        sampled_next = jax.vmap(lambda k: dynamics.predict_next_state(k, s_c, a))(keys[1:])
        mean_sampled_next = jnp.mean(sampled_next, axis=0)
        real_delta = s_n - s_c

        print(
            f"  step {i}: log_prob(real s_next)={lp:.2f} | "
            f"real next={np.array(s_n)} | "
            f"model mean next={np.array(mean_sampled_next)}"
        )

In [11]:
import numpy as np
def evaluate_actor_in_env(
    actor: FlowActor,
    env: gym.Env,
    key: jax.Array,
    max_steps: int = 1000,
) -> tuple[float, int, jax.Array, jax.Array, jax.Array, jax.Array]:
    """Rolls the actor out in the *real* environment (no learned dynamics).

    Returns (total_reward, episode_length, s_curr_ep, action_ep, s_next_ep)
    where the last three are explicit, self-contained transition triplets
    from this single episode -- safe to append directly to a multi-episode
    dynamics buffer without any boundary-alignment bugs.
    """
    curr_state, _ = env.reset()
    prev_state = curr_state  # no history yet; use s_0 as its own "previous" state

    s_curr_ep: list[np.ndarray] = []
    action_ep: list[np.ndarray] = []
    s_next_ep: list[np.ndarray] = []
    rewards: list[np.ndarray] = []

    total_reward = 0.0
    for _ in range(max_steps):
        key, ak = jax.random.split(key)
        action = actor.sample_action(ak, jnp.array(prev_state), jnp.array(curr_state))
        action_np = np.array(action)

        next_state, reward, terminated, truncated, _ = env.step(action_np)
        rewards.append(float(reward))
        total_reward += float(reward)

        s_curr_ep.append(curr_state)
        action_ep.append(action_np)
        s_next_ep.append(next_state)

        prev_state, curr_state = curr_state, next_state
        if terminated or truncated:
            break

    episode_length = len(action_ep)
    return (
        total_reward,
        episode_length,
        jnp.array(s_curr_ep),
        jnp.array(action_ep),
        jnp.array(s_next_ep),
        jnp.array(rewards)
    )

In [12]:
SUCCESS_LENGTH = 950  # out of max_episode_steps=1000 for InvertedPendulum-v5
MAX_OUTER_ITERS = 5

In [13]:
first_dynamics = dynamics
first_actor = actor
first_reward = reward_fn

In [14]:
# dynamics = first_dynamics
# actor = first_actor
# reward_fn = first_reward

In [15]:
buf_states_for_actor_init = states
s_curr_buf = states[:-1]
action_buf = actions[:-1]
s_next_buf = states[1:]
curr_rewards = rewards[:-1]

for outer_iter in range(MAX_OUTER_ITERS):
    print(f"\n=== Outer iteration {outer_iter} ===")

    key, dyn_key, actor_key, eval_key, reality_key, rew_key = jax.random.split(key, 6)

    print("Refitting dynamics model on full transition buffer...")
    dynamics, _ = train_flow(s_curr_buf, action_buf, s_next_buf, dyn_key, flow=dynamics)

    print("Training actor against updated dynamics model...")
    actor = train_actor(actor, dynamics, buf_states_for_actor_init, actor_key, optim, reward_fn, num_train_steps=250)

    print("Evaluating actor in the real environment...")
    total_reward, episode_length, s_curr_ep, action_ep, s_next_ep, episode_rewards = evaluate_actor_in_env(actor, env, eval_key)
    print(f"Real-env return: {total_reward:.1f}, episode length: {episode_length}")

    print("Model-reality gap on this episode's real transitions:")
    diagnose_model_reality_gap(dynamics, s_curr_ep, action_ep, s_next_ep, key=reality_key)
    
    # Fold the on-policy transitions back into the buffers regardless of
    # success/failure -- this is the data that corrects the dynamics model
    # in the state-action regions the actor actually visits. Each episode's
    # triplets are self-contained, so concatenation across episodes never
    # fabricates a transition.
    s_curr_buf = jnp.concatenate([s_curr_buf, s_curr_ep], axis=0)
    action_buf = jnp.concatenate([action_buf, jnp.atleast_2d(action_ep)], axis=0)
    s_next_buf = jnp.concatenate([s_next_buf, s_next_ep], axis=0)
    curr_rewards = jnp.concatenate([curr_rewards, episode_rewards[:,jnp.newaxis]], axis=0)
    buf_states_for_actor_init = jnp.concatenate([buf_states_for_actor_init, s_curr_ep], axis=0)

    # Update learned reward function
    reward_fn = train_reward(s_curr_buf, action_buf, curr_rewards, rew_key)
    
    if episode_length >= SUCCESS_LENGTH:
        print(f"Success: balanced for {episode_length} steps. Stopping.")
        break
else:
    print(
        "Reached MAX_OUTER_ITERS without success; consider more iterations, "
        "a longer rollout horizon, or a stronger reward surrogate."
    )


=== Outer iteration 0 ===
Refitting dynamics model on full transition buffer...


 12%|███▉                             | 119/1000 [00:09<01:06, 13.21it/s, train=4.76, val=4.45 (Max patience reached)]


Training actor against updated dynamics model...
  [actor] step 0, loss -200.9423
  [actor] step 20, loss -200.9687
  [actor] step 40, loss -200.9975
  [actor] step 60, loss -200.9469
  [actor] step 80, loss -200.9464
  [actor] step 100, loss -200.9134
  [actor] step 120, loss -200.9706
  [actor] step 140, loss -200.9328
  [actor] step 160, loss -200.8974
  [actor] step 180, loss -200.9111
  [actor] step 200, loss -200.8862
  [actor] step 220, loss -200.9506
  [actor] step 240, loss -200.9004
Evaluating actor in the real environment...
Real-env return: 63.2, episode length: 8
Model-reality gap on this episode's real transitions:
  step 0: log_prob(real s_next)=-34.35 | real next=[-0.045168    0.07604741 -0.09303212  0.9971042   0.99566311 -0.87735162
  1.91062439 -2.48576544  0.        ] | model mean next=[-4.19587113e-02  8.80912109e-02 -8.05197897e-02  1.99650241e+00
  1.99464163e+00  2.69785184e-01 -2.05828147e-01  3.38579867e-01
 -2.84827452e-04]
  step 1: log_prob(real s_next)=7.8

  0%|          | 0/1600 [00:00<?, ?it/s]


=== Outer iteration 1 ===
Refitting dynamics model on full transition buffer...


...


=== Outer iteration 4 ===
Refitting dynamics model on full transition buffer...


  7%|██▎                             | 71/1000 [00:05<01:11, 12.96it/s, train=-11.8, val=-1.09 (Max patience reached)]


Training actor against updated dynamics model...
  [actor] step 0, loss -83.5071
  [actor] step 20, loss -81.9992
  [actor] step 40, loss -82.8835
  [actor] step 60, loss -82.6599
  [actor] step 80, loss -82.8236
  [actor] step 100, loss -82.7190
  [actor] step 120, loss -81.0498
  [actor] step 140, loss -81.5537
  [actor] step 160, loss -82.5940
  [actor] step 180, loss -81.4369
  [actor] step 200, loss -83.5384
  [actor] step 220, loss -81.6139
  [actor] step 240, loss -82.6238
Evaluating actor in the real environment...
Real-env return: 44.5, episode length: 6
Model-reality gap on this episode's real transitions:
  step 0: log_prob(real s_next)=4.61 | real next=[-0.08784504  0.06127439  0.06389197  0.99812096  0.99795682  0.21959135
 -0.68505624  0.97199944  0.        ] | model mean next=[-1.92431480e-01  1.70792325e-01  5.77651992e-02  1.98400435e+00
  2.02787635e+00 -6.03000242e-01 -3.22871448e-01 -9.88765649e-02
 -8.31100427e-04]
  step 1: log_prob(real s_next)=3.56 | real next=[

  0%|          | 0/1600 [00:00<?, ?it/s]

Reached MAX_OUTER_ITERS without success; consider more iterations, a longer rollout horizon, or a stronger reward surrogate.


In [16]:
rewards.shape

(4096, 1)

In [17]:
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

In [18]:
env = gym.make("InvertedDoublePendulum-v5", render_mode="rgb_array")

# 2. Wrap environment to record videos into a folder
env = RecordVideo(
    env, 
    video_folder="./gym_videos", 
    episode_trigger=lambda x: True  # Record every episode
)

# 3. Run your controller loop
observation, info = env.reset()
done = False
prev_state = observation
curr_state = observation
while not done:
    # --- INSERT YOUR TRAINED CONTROLLER HERE ---
    key, ak = jax.random.split(key)
    action = actor.sample_action(ak, jnp.array(prev_state), jnp.array(curr_state))
    # ------------------------------------------
    
    observation, reward, terminated, truncated, info = env.step(action)
    prev_state = curr_state
    curr_state = observation
    done = terminated or truncated

# Close the environment to finalize and save the video file
env.close()

# 4. Display the recorded video inline
# RecordVideo automatically names the file based on the episode index
video_path = "./gym_videos/rl-video-episode-0.mp4"
display(Video(video_path, embed=True, width=600))